### Temporal Attention Network (TAN) Implementation

The **Temporal Attention Network (TAN)** incorporates `attention mechanisms` in `Recurrent Neural Networks (RNNs)` to focus more on important regions or frames within a `video`.

An attention layer will be integrated within an LSTM model. The model will first extract features from video frames using a pre-trained CNN like ResNet50 or VGG16, and then it will apply an attention mechanism over the sequence of features before feeding them into an LSTM for the temporal analysis.

### Key improvements for higher performance:

- Uses EfficientNetB0 instead of ResNet50 for better efficiency  
- Implements Multi-Head Attention mechanism  
- Uses Bidirectional LSTM layers   
- Adds residual connections  
- Implements AdamW optimizer with weight decay  
- Uses GELU activation for better gradient flow  
- Implements advanced frame sampling strategy  
- Adds comprehensive data preprocessing  
- Uses learning rate scheduling  
- Implements proper train/validation/test split  

# GOOD

In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GlobalAveragePooling2D, 
    MultiHeadAttention, LayerNormalization, Dropout,
    Add, Concatenate, Bidirectional
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, 
    ReduceLROnPlateau, TensorBoard
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s') 

logger = logging.getLogger(__name__)

In [3]:
class DataProcessor:
    def __init__(self, max_frames=10, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        logger.info(f"Initialized DataProcessor with max_frames={max_frames}, frame_size={frame_size}")

    def extract_frames(self, video_path, max_frames=10):
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        frames = []
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            logger.error(f"Failed to open video file: {video_path}")
            return None

        try:
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames == 0:
                logger.error(f"Video has no frames: {video_path}")
                return None

            sample_interval = max(1, total_frames // max_frames)
            frame_count = 0
            sampled_count = 0

            while sampled_count < max_frames:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_count % sample_interval == 0:
                    frame = cv2.resize(frame, self.frame_size)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
                    frames.append(frame)
                    sampled_count += 1

                frame_count += 1

            cap.release()

            # Pad with zeros if necessary
            while len(frames) < max_frames:
                frames.append(np.zeros((self.frame_size[0], self.frame_size[1], 3), dtype=np.uint8))

            frames_array = np.array(frames)
            logger.info(f"Successfully extracted {len(frames)} frames from {video_path}")
            return frames_array

        except Exception as e:
            logger.error(f"Error extracting frames from {video_path}: {str(e)}")
            cap.release()
            return None

    def process_dataset(self, video_dir, metadata_path, max_frames):
        if not os.path.exists(metadata_path):
            logger.error(f"Metadata file not found: {metadata_path}")
            return None, None, None

        if not os.path.exists(video_dir):
            logger.error(f"Video directory not found: {video_dir}")
            return None, None, None

        try:
            metadata = pd.read_csv(metadata_path) if metadata_path.endswith('.csv') else pd.read_excel(metadata_path)
            logger.info(f"Loaded metadata with {len(metadata)} entries")

            if 'HowFunny' not in metadata.columns or 'VideoID' not in metadata.columns:
                logger.error("Required columns 'HowFunny' or 'VideoID' not found in metadata")
                return None, None, None

            metadata = metadata.dropna(subset=['HowFunny'])
            metadata['HowFunny'] = pd.Categorical(metadata['HowFunny'])
            metadata['HowFunny_Code'] = metadata['HowFunny'].cat.codes

            value_counts = metadata['HowFunny_Code'].value_counts()
            valid_classes = value_counts[value_counts >= 2].index
            metadata = metadata[metadata['HowFunny_Code'].isin(valid_classes)]

            processed_data = []
            labels = []
            processed_videos = []

            total_videos = len(metadata)
            logger.info(f"Processing {total_videos} videos...")

            for idx, row in metadata.iterrows():
                video_path = os.path.join(video_dir, row['VideoID'])
                if not any(video_path.endswith(ext) for ext in ['.mp4', '.avi', '.mov']):
                    video_path += '.mp4'

                if os.path.exists(video_path):
                    frames = self.extract_frames(video_path, max_frames)
                    if frames is not None and frames.shape == (max_frames, *self.frame_size, 3):
                        processed_data.append(frames)
                        labels.append(row['HowFunny_Code'])
                        processed_videos.append(row['VideoID'])
                    else:
                        logger.warning(f"Skipped invalid frames from {video_path}")
                else:
                    logger.warning(f"Video not found: {video_path}")

                if idx % 10 == 0:
                    logger.info(f"Processed {idx+1}/{total_videos} videos")

            if not processed_data:
                logger.error("No valid videos were processed")
                return None, None, None

            processed_data = np.array(processed_data)
            labels = np.array(labels)
            logger.info(f"Successfully processed {len(processed_data)} videos")
            logger.info(f"Data shape: {processed_data.shape}, Labels shape: {labels.shape}")
            
            return processed_data, labels, processed_videos

        except Exception as e:
            logger.error(f"Error processing dataset: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None, None, None

In [4]:
class FeatureExtractor:
    def __init__(self, input_shape=(224, 224, 3)):
        logger.info("Initializing FeatureExtractor...")
        self.model = self._build_extractor(input_shape)

    def _build_extractor(self, input_shape):
        try:
            base_model = EfficientNetB0(
                weights='imagenet',
                include_top=False,
                input_shape=input_shape
            )
            for layer in base_model.layers[:-10]:
                layer.trainable = False

            model = Model(
                inputs=base_model.input,
                outputs=GlobalAveragePooling2D()(base_model.output)
            )
            logger.info("Feature extractor model built successfully")
            return model

        except Exception as e:
            logger.error(f"Error building feature extractor: {str(e)}")
            raise

    def extract_features(self, frames):
        try:
            features = []
            for frame in frames:
                frame = frame.astype('float32') / 255.0  # Normalize pixel values
                frame_features = self.model.predict(
                    np.expand_dims(frame, axis=0),
                    verbose=0
                )
                features.append(frame_features.flatten())
            return np.array(features)

        except Exception as e:
            logger.error(f"Error extracting features: {str(e)}")
            return None

    def process_dataset(self, frames_data):
        if frames_data is None:
            logger.error("Input frames_data is None")
            return None

        try:
            features_list = []
            total_videos = len(frames_data)
            logger.info(f"Processing features for {total_videos} videos")

            for i, frames in enumerate(frames_data):
                features = self.extract_features(frames)
                if features is not None:
                    features_list.append(features)
                
                if (i + 1) % 10 == 0:
                    logger.info(f"Processed features for {i+1}/{total_videos} videos")

            if not features_list:
                logger.error("No features were successfully extracted")
                return None

            features_array = np.array(features_list)
            logger.info(f"Feature extraction complete. Output shape: {features_array.shape}")
            return features_array

        except Exception as e:
            logger.error(f"Error in feature extraction: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None



In [5]:
class ModelTrainer:
    def __init__(self, input_shape, num_classes, num_attention_heads=8, learning_rate=1e-4):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.num_attention_heads = num_attention_heads
        self.learning_rate = learning_rate 
        
        logger.info(f"Initializing ModelTrainer with {num_classes} classes")
        self.model = self._build_model()

    def _build_model(self):
        try:
            inputs = Input(shape=self.input_shape)
            
            # Bidirectional LSTM with residual connection
            lstm_out = Bidirectional(LSTM(256, return_sequences=True))(inputs)
            lstm_out = LayerNormalization()(lstm_out)
            lstm_out = Dropout(0.3)(lstm_out)
            
            # Multi-head attention
            attention_out = MultiHeadAttention(
                num_heads=self.num_attention_heads,
                key_dim=32
            )(lstm_out, lstm_out)
            
            x = Add()([lstm_out, attention_out])
            x = LayerNormalization()(x)
            
            lstm_out2 = Bidirectional(LSTM(128))(x)
            lstm_out2 = LayerNormalization()(lstm_out2)
            lstm_out2 = Dropout(0.2)(lstm_out2)
            
            dense1 = Dense(256, activation='gelu')(lstm_out2)
            dense1 = LayerNormalization()(dense1)
            dense1 = Dropout(0.2)(dense1)
            
            dense2 = Dense(128, activation='gelu')(dense1)
            dense2 = LayerNormalization()(dense2)
            dense2 = Dropout(0.2)(dense2)
            
            outputs = Dense(self.num_classes, activation='softmax')(dense2)
            
            model = Model(inputs, outputs)
            
            optimizer = AdamW(
                learning_rate=self.learning_rate,
                weight_decay=1e-5
            )
            
            model.compile(
                optimizer=optimizer,
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            
            logger.info("Model built successfully")
            model.summary()
            return model

        except Exception as e:
            logger.error(f"Error building model: {str(e)}")
            raise

    def train(self, X, y, epochs=20, batch_size=32):
        if X is None or y is None:
            logger.error("Training data or labels are None")
            return None

        try:
            logger.info("Splitting dataset into train/validation/test sets")
            X_temp, X_test, y_temp, y_test = train_test_split(
                X, y, test_size=0.2, stratify=y, random_state=42
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42
            )
            
            logger.info(f"Training set shape: {X_train.shape}")
            logger.info(f"Validation set shape: {X_val.shape}")
            logger.info(f"Test set shape: {X_test.shape}")

            # Create output directory for model artifacts
            os.makedirs('model_outputs', exist_ok=True)
            
            callbacks = [
                EarlyStopping(
                    monitor='val_accuracy',
                    patience=15,
                    restore_best_weights=True,
                    verbose=1
                ),
                ModelCheckpoint(
                    'model_outputs/best_model.keras',
                    monitor='val_accuracy',
                    save_best_only=True,
                    verbose=1
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.2,
                    patience=5,
                    min_lr=1e-6,
                    verbose=1
                ),
                TensorBoard(log_dir='model_outputs/logs')
            ]
            
            logger.info("Starting model training...")
            history = self.model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                callbacks=callbacks,
                verbose=1
            )
            
            logger.info("Training completed. Evaluating model...")
            self.evaluate(X_test, y_test, history)
            return history

        except Exception as e:
            logger.error(f"Error during training: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None

    def evaluate(self, X_test, y_test, history):
        try:
            test_loss, test_accuracy = self.model.evaluate(X_test, y_test, verbose=1)
            logger.info(f'Test Loss: {test_loss:.4f}')
            logger.info(f'Test Accuracy: {test_accuracy:.4f}')
            
            y_pred = self.model.predict(X_test)
            y_pred_classes = np.argmax(y_pred, axis=1)
            
            self._plot_results(history, y_test, y_pred_classes)
            
            report = classification_report(y_test, y_pred_classes)
            logger.info("\nClassification Report:\n" + report)
            
            with open('model_outputs/classification_report.txt', 'w') as f:
                f.write(report)

        except Exception as e:
            logger.error(f"Error during evaluation: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())

    def _plot_results(self, history, y_test, y_pred_classes):
        try:
            plt.figure(figsize=(15, 5))
            
            # Accuracy plot
            plt.subplot(1, 3, 1)
            plt.plot(history.history['accuracy'], label='Train')
            plt.plot(history.history['val_accuracy'], label='Validation')
            plt.title('Model Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.legend()
            
            # Loss plot
            plt.subplot(1, 3, 2)
            plt.plot(history.history['loss'], label='Train')
            plt.plot(history.history['val_loss'], label='Validation')
            plt.title('Model Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            # Confusion matrix
            plt.subplot(1, 3, 3)
            cm = confusion_matrix(y_test, y_pred_classes)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title('Confusion Matrix')
            plt.xlabel('Predicted')
            plt.ylabel('True')
            
            plt.tight_layout()
            plt.savefig('model_outputs/training_results.png')
            plt.close()
            
            logger.info("Training visualizations saved to model_outputs/training_results.png")

        except Exception as e:
            logger.error(f"Error creating plots: {str(e)}")

def main():
    try:
        # Initialize processors
        data_processor = DataProcessor(max_frames=10, frame_size=(224, 224))
        feature_extractor = FeatureExtractor()

        # Process data
        logger.info("Starting data processing...")
        frames, labels, videos = data_processor.process_dataset(
            video_dir='data/videos',
            metadata_path='data/excel.xlsx',
            max_frames=10
        )

        if frames is None or labels is None:
            logger.error("Data processing failed")
            return

        # Extract features
        logger.info("Starting feature extraction...")
        features = feature_extractor.process_dataset(frames)

        if features is None:
            logger.error("Feature extraction failed")
            return

        # Initialize and train model
        logger.info("Starting model training...")
        trainer = ModelTrainer(
            input_shape=(features.shape[1], features.shape[2]),
            num_classes=len(np.unique(labels))
        )

        history = trainer.train(
            features, labels,
            epochs=20,
            batch_size=16
        )

        if history is None:
            logger.error("Model training failed")
            return

        logger.info("Pipeline completed successfully")

    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())

if __name__ == "__main__":
    main()

2024-12-13 01:25:04,355 - INFO - Initialized DataProcessor with max_frames=10, frame_size=(224, 224)
2024-12-13 01:25:04,356 - INFO - Initializing FeatureExtractor...
2024-12-13 01:25:04,792 - INFO - Feature extractor model built successfully
2024-12-13 01:25:04,793 - INFO - Starting data processing...
2024-12-13 01:25:05,069 - INFO - Loaded metadata with 1426 entries
2024-12-13 01:25:05,073 - INFO - Processing 1424 videos...
2024-12-13 01:25:05,101 - INFO - Successfully extracted 10 frames from data/videos/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4
2024-12-13 01:25:05,102 - INFO - Processed 1/1424 videos
2024-12-13 01:25:05,194 - INFO - Successfully extracted 10 frames from data/videos/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4
2024-12-13 01:25:05,267 - INFO - Successfully extracted 10 frames from data/videos/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4
2024-12-13 01:25:05,310 - INFO - Successfully extracted 10 frames from data/videos/NFCK3A.joke1.rep3.take1.ThatsNotAHat.mp4
2024-12-13 01:25:

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 10, 1280)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 10, 512)   │  3,147,776 │ input_layer_1[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 512)   │      1,024 │ bidirectional[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 512)   │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 512)   │    525,568 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 512)   │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 512)   │      1,024 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 256)       │    656,384 │ layer_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256)       │        512 │ bidirectional_1[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     65,792 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 256)       │        512 │ dense[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128)       │        256 │ dense_1[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 5)         │        645 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,432,389 (16.91 MB)

 Trainable params: 4,432,389 (16.91 MB)

 Non-trainable params: 0 (0.00 B)

... (3 outputs left out)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3815 - loss: 1.2505


2024-12-13 01:41:15,095 - INFO - Test Loss: 1.2154
2024-12-13 01:41:15,095 - INFO - Test Accuracy: 0.3860


9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step


2024-12-13 01:41:16,020 - INFO - Training visualizations saved to model_outputs/training_results.png
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/videop/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

# With Data Augumentation

In [30]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GlobalAveragePooling2D, 
    MultiHeadAttention, LayerNormalization, Dropout,
    Add, Concatenate, Bidirectional
)
from tensorflow.keras.optimizers import AdamW,SGD,Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, 
    ReduceLROnPlateau, TensorBoard
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [31]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [32]:
class DataAugmentor:
    def __init__(self):
        self.augmenter = A.Compose([
            A.RandomBrightnessContrast(p=0.4),
            A.RandomGamma(p=0.4),
            A.GaussNoise(p=0.2),
            A.HorizontalFlip(p=0.5),
            A.OneOf([
                A.MotionBlur(p=0.2),
                A.MedianBlur(blur_limit=2, p=0.1),
                A.GaussianBlur(blur_limit=2, p=0.1),
            ], p=0.2),
            A.OneOf([
                A.OpticalDistortion(p=0.2),
                A.GridDistortion(p=0.1),
            ], p=0.2),
            A.OneOf([
                A.CLAHE(clip_limit=2),
                A.Sharpen(),
                A.Emboss(),
            ], p=0.3),
        ])

    def augment_frame(self, frame):
        try:
            augmented = self.augmenter(image=frame)
            return augmented['image']
        except Exception as e:
            logger.error(f"Error in frame augmentation: {str(e)}")
            return frame

    def augment_video_frames(self, frames):
        try:
            augmented_frames = []
            for frame in frames:
                augmented_frame = self.augment_frame(frame)
                augmented_frames.append(augmented_frame)
            return np.array(augmented_frames)
        except Exception as e:
            logger.error(f"Error in video augmentation: {str(e)}")
            return frames

In [33]:
class DataProcessor:
    
    def __init__(self, max_frames=10, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.augmentor = DataAugmentor()
        logger.info(f"Initialized DataProcessor with max_frames={max_frames}, frame_size={frame_size}")

    def extract_frames(self, video_path, max_frames=10, augment=False):
        if not os.path.exists(video_path):
            logger.error(f"Video file not found: {video_path}")
            return None

        frames = []
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            logger.error(f"Failed to open video file: {video_path}")
            return None

        try:
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames == 0:
                logger.error(f"Video has no frames: {video_path}")
                return None

            sample_interval = max(1, total_frames // max_frames)
            frame_count = 0
            sampled_count = 0

            while sampled_count < max_frames:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_count % sample_interval == 0:
                    frame = cv2.resize(frame, self.frame_size)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    
                    if augment:
                        frame = self.augmentor.augment_frame(frame)
                    
                    frames.append(frame)
                    sampled_count += 1

                frame_count += 1

            cap.release()

            while len(frames) < max_frames:
                frames.append(np.zeros((self.frame_size[0], self.frame_size[1], 3), dtype=np.uint8))

            frames_array = np.array(frames)
            logger.info(f"Successfully extracted {len(frames)} frames from {video_path}")
            return frames_array

        except Exception as e:
            logger.error(f"Error extracting frames from {video_path}: {str(e)}")
            cap.release()
            return None

    def process_dataset(self, video_dir, metadata_path, max_frames, augment=True):
        if not os.path.exists(metadata_path):
            logger.error(f"Metadata file not found: {metadata_path}")
            return None, None, None

        if not os.path.exists(video_dir):
            logger.error(f"Video directory not found: {video_dir}")
            return None, None, None

        try:
            metadata = pd.read_csv(metadata_path) if metadata_path.endswith('.csv') else pd.read_excel(metadata_path)
            logger.info(f"Loaded metadata with {len(metadata)} entries")

            if 'HowFunny' not in metadata.columns or 'VideoID' not in metadata.columns:
                logger.error("Required columns 'HowFunny' or 'VideoID' not found in metadata")
                return None, None, None

            metadata = metadata.dropna(subset=['HowFunny'])
            metadata['HowFunny'] = pd.Categorical(metadata['HowFunny'])
            metadata['HowFunny_Code'] = metadata['HowFunny'].cat.codes

            value_counts = metadata['HowFunny_Code'].value_counts()
            valid_classes = value_counts[value_counts >= 2].index
            metadata = metadata[metadata['HowFunny_Code'].isin(valid_classes)]

            processed_data = []
            augmented_data = []
            labels = []
            processed_videos = []

            total_videos = len(metadata)
            logger.info(f"Processing {total_videos} videos...")

            for idx, row in metadata.iterrows():
                video_path = os.path.join(video_dir, row['VideoID'])
                if not any(video_path.endswith(ext) for ext in ['.mp4', '.avi', '.mov']):
                    video_path += '.mp4'

                if os.path.exists(video_path):
                    # Process original frames
                    frames = self.extract_frames(video_path, max_frames, augment=False)
                    if frames is not None and frames.shape == (max_frames, *self.frame_size, 3):
                        processed_data.append(frames)
                        labels.append(row['HowFunny_Code'])
                        processed_videos.append(row['VideoID'])

                        # Add augmented version if augmentation is enabled
                        if augment:
                            augmented_frames = self.augmentor.augment_video_frames(frames)
                            augmented_data.append(augmented_frames)
                            labels.append(row['HowFunny_Code'])
                            processed_videos.append(f"{row['VideoID']}_augmented")
                    else:
                        logger.warning(f"Skipped invalid frames from {video_path}")
                else:
                    logger.warning(f"Video not found: {video_path}")

                if idx % 10 == 0:
                    logger.info(f"Processed {idx+1}/{total_videos} videos")

            if not processed_data:
                logger.error("No valid videos were processed")
                return None, None, None

            # Combine original and augmented data
            if augment and augmented_data:
                processed_data.extend(augmented_data)

            processed_data = np.array(processed_data)
            labels = np.array(labels)
            logger.info(f"Successfully processed {len(processed_data)} videos (including augmentations)")
            logger.info(f"Data shape: {processed_data.shape}, Labels shape: {labels.shape}")
            
            return processed_data, labels, processed_videos

        except Exception as e:
            logger.error(f"Error processing dataset: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None, None, None

In [34]:
class FeatureExtractor: 
    
    def __init__(self, input_shape=(224, 224, 3)):
        logger.info("Initializing FeatureExtractor...")
        self.model = self._build_extractor(input_shape)

    def _build_extractor(self, input_shape):
        try:
            base_model = EfficientNetB0(
                weights='imagenet',
                include_top=False,
                input_shape=input_shape
            )
            for layer in base_model.layers[:-10]:
                layer.trainable = False

            model = Model(
                inputs=base_model.input,
                outputs=GlobalAveragePooling2D()(base_model.output)
            )
            logger.info("Feature extractor model built successfully")
            return model

        except Exception as e:
            logger.error(f"Error building feature extractor: {str(e)}")
            raise

    def extract_features(self, frames):
        try:
            features = []
            for frame in frames:
                frame = frame.astype('float32') / 255.0  # Normalize pixel values
                frame_features = self.model.predict(
                    np.expand_dims(frame, axis=0),
                    verbose=0
                )
                features.append(frame_features.flatten())
            return np.array(features)

        except Exception as e:
            logger.error(f"Error extracting features: {str(e)}")
            return None

    def process_dataset(self, frames_data):
        if frames_data is None:
            logger.error("Input frames_data is None")
            return None

        try:
            features_list = []
            total_videos = len(frames_data)
            logger.info(f"Processing features for {total_videos} videos")

            for i, frames in enumerate(frames_data):
                features = self.extract_features(frames)
                if features is not None:
                    features_list.append(features)
                
                if (i + 1) % 10 == 0:
                    logger.info(f"Processed features for {i+1}/{total_videos} videos")

            if not features_list:
                logger.error("No features were successfully extracted")
                return None

            features_array = np.array(features_list)
            logger.info(f"Feature extraction complete. Output shape: {features_array.shape}")
            return features_array

        except Exception as e:
            logger.error(f"Error in feature extraction: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None

In [35]:
class ModelTrainer:
    
    def __init__(self, input_shape, num_classes, num_attention_heads=8, learning_rate=1e-4):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.num_attention_heads = num_attention_heads
        self.learning_rate = learning_rate
        logger.info(f"Initializing ModelTrainer with {num_classes} classes")
        self.model = self._build_model()

    def _build_model(self):
        try:
            inputs = Input(shape=self.input_shape)
            
            # Bidirectional LSTM with residual connection
            lstm_out = Bidirectional(LSTM(512, return_sequences=True))(inputs)
            lstm_out = LayerNormalization()(lstm_out)
            lstm_out = Dropout(0.3)(lstm_out)
            
            # Multi-head attention
            attention_out = MultiHeadAttention(
                num_heads=self.num_attention_heads,
                key_dim=32
            )(lstm_out, lstm_out)
            
            x = Add()([lstm_out, attention_out])
            x = LayerNormalization()(x)
            
            lstm_out2 = Bidirectional(LSTM(256))(x)
            lstm_out2 = LayerNormalization()(lstm_out2)
            lstm_out2 = Dropout(0.3)(lstm_out2)
            
            dense1 = Dense(256, activation='relu')(lstm_out2)
            dense1 = LayerNormalization()(dense1)
            dense1 = Dropout(0.1)(dense1)
            
            dense2 = Dense(128, activation='relu')(dense1)
            dense2 = LayerNormalization()(dense2)
            dense2 = Dropout(0.1)(dense2)
            
            outputs = Dense(self.num_classes, activation='softmax')(dense2)
            
            model = Model(inputs, outputs)
            
            optimizer = SGD(
                learning_rate=self.learning_rate,
                weight_decay=1e-4
            )
            
            model.compile(
                optimizer=optimizer,
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            
            logger.info("Model built successfully")
            model.summary()
            return model

        except Exception as e:
            logger.error(f"Error building model: {str(e)}")
            raise

    def train(self, X, y, epochs=100, batch_size=32):
        if X is None or y is None:
            logger.error("Training data or labels are None")
            return None

        try:
            logger.info("Splitting dataset into train/validation/test sets")
            X_temp, X_test, y_temp, y_test = train_test_split(
                X, y, test_size=0.2, stratify=y, random_state=42
            )
            X_train, X_val, y_train, y_val = train_test_split(
                X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42
            )
            
            logger.info(f"Training set shape: {X_train.shape}")
            logger.info(f"Validation set shape: {X_val.shape}")
            logger.info(f"Test set shape: {X_test.shape}")

            # Create output directory for model artifacts
            os.makedirs('model_outputs', exist_ok=True)
            
            callbacks = [
                EarlyStopping(
                    monitor='val_accuracy',
                    patience=20,
                    restore_best_weights=True,
                    verbose=1
                ),
                ModelCheckpoint(
                    'model_outputs/best_model.keras',
                    monitor='val_accuracy',
                    save_best_only=True,
                    verbose=1
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.2,
                    patience=5,
                    min_lr=1e-4,
                    verbose=1
                ),
                TensorBoard(log_dir='model_outputs/logs')
            ]
            
            logger.info("Starting model training...")
            history = self.model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                callbacks=callbacks,
                verbose=1
            )
            
            logger.info("Training completed. Evaluating model...")
            self.evaluate(X_test, y_test, history)
            return history

        except Exception as e:
            logger.error(f"Error during training: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            return None

    def evaluate(self, X_test, y_test, history):
        try:
            test_loss, test_accuracy = self.model.evaluate(X_test, y_test, verbose=1)
            logger.info(f'Test Loss: {test_loss:.4f}')
            logger.info(f'Test Accuracy: {test_accuracy:.4f}')
            
            y_pred = self.model.predict(X_test)
            y_pred_classes = np.argmax(y_pred, axis=1)
            
            self._plot_results(history, y_test, y_pred_classes)
            
            report = classification_report(y_test, y_pred_classes)
            logger.info("\nClassification Report:\n" + report)
            
            with open('model_outputs/classification_report.txt', 'w') as f:
                f.write(report)

        except Exception as e:
            logger.error(f"Error during evaluation: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())

    def _plot_results(self, history, y_test, y_pred_classes):
        try:
            plt.figure(figsize=(15, 5))
            
            # Accuracy plot
            plt.subplot(1, 3, 1)
            plt.plot(history.history['accuracy'], label='Train')
            plt.plot(history.history['val_accuracy'], label='Validation')
            plt.title('Model Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.legend()
            
            # Loss plot
            plt.subplot(1, 3, 2)
            plt.plot(history.history['loss'], label='Train')
            plt.plot(history.history['val_loss'], label='Validation')
            plt.title('Model Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            # Confusion matrix
            plt.subplot(1, 3, 3)
            cm = confusion_matrix(y_test, y_pred_classes)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title('Confusion Matrix')
            plt.xlabel('Predicted')
            plt.ylabel('True')
            
            plt.tight_layout()
            plt.savefig('model_outputs/training_results.png')
            plt.close()
            
            logger.info("Training visualizations saved to model_outputs/training_results.png")

        except Exception as e:
            logger.error(f"Error creating plots: {str(e)}") 

In [36]:
def main(): 
    
    try:
        # Initialize processors 
        data_processor = DataProcessor(max_frames=20, frame_size=(224, 224))
        feature_extractor = FeatureExtractor()

        # Process data 
        logger.info("Starting data processing...")
        frames, labels, videos = data_processor.process_dataset(
            video_dir='data/main_data/videos',
            metadata_path='/Users/jeremiah/phd_thesis/data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx',
            max_frames=20,
            augment=True  # Enable data augmentation
        )

        if frames is None or labels is None:
            logger.error("Data processing failed")
            return

        # Extract features
        logger.info("Starting feature extraction...")
        features = feature_extractor.process_dataset(frames)

        if features is None:
            logger.error("Feature extraction failed")
            return

        # Initialize and train model
        logger.info("Starting model training...")
        trainer = ModelTrainer(
            input_shape=(features.shape[1], features.shape[2]),
            num_classes=len(np.unique(labels))
        )

        history = trainer.train(
            features, labels,
            epochs=100,
            batch_size=16
        )

        if history is None:
            logger.error("Model training failed")
            return

        logger.info("Pipeline completed successfully")

    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())

if __name__ == "__main__":
    main()

2025-01-30 10:38:26,484 - ERROR - Pipeline failed: 1 validation error for InitSchema
blur_limit
  Value error, Blur limit must be 0 or odd. Got: (2, 3) [type=value_error, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error
2025-01-30 10:38:26,488 - ERROR - Traceback (most recent call last):
  File "/var/folders/fp/nfh3jlt16mv7dm_xv5cq2c040000gn/T/ipykernel_92608/2523041571.py", line 5, in main
    data_processor = DataProcessor(max_frames=20, frame_size=(224, 224))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/fp/nfh3jlt16mv7dm_xv5cq2c040000gn/T/ipykernel_92608/2846081014.py", line 6, in __init__
    self.augmentor = DataAugmentor()
                     ^^^^^^^^^^^^^^^
  File "/var/folders/fp/nfh3jlt16mv7dm_xv5cq2c040000gn/T/ipykernel_92608/2408865643.py", line 10, in __init__
    A.MedianBlur(blur_limit=2, p=0.1),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs